# 02- Construção das tabelas dimensões
---
## Objetivo

Este notebook tem como objetivo construir as tabelas dimensão utilizadas no modelo dimensional do projeto de Business Intelligence sobre assistência estudantil da UFPB.

As dimensões são obtidas a partir das consultas realizadas no SEDAP+ e passam por etapas de verificação, tratamento e padronização antes de serem utilizadas na construção da tabela fato e do dashboard.

 ---

## Importação das bibliotecas necessárias

In [1]:
import sys
from pathlib import Path
import pandas as pd

In [2]:
# Define a pasta raiz do projeto (um nível acima da pasta /notebooks)
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Adiciona a raiz ao sys.path para o Python encontrar a pasta 'src'
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

# Caminhos dos arquivos de entrada e saída
PATH_RAW_CURSOS = BASE_DIR / "data" / "raw" / "INEP" / "Censo da Educação Superior" / "2023" / "dados" / "MICRODADOS_CADASTRO_CURSOS_2023.CSV"
PATH_RAW_SEDAP = BASE_DIR / "data" / "raw" / "SEDAP" / "dim_curso.csv"
PATH_PROCESSED_CSV = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_curso_ufpb_campus_1.csv"

 ---
# 1. Construção da Dimensão Curso
## 1.1 Carregar os dados

In [4]:
import importlib
import src.dimensions.dim_curso as dc

importlib.reload(dc)
from src.dimensions.dim_curso_campus1 import criar_dim_curso

# 1. Instancia a Dimensão Curso
dim_curso = criar_dim_curso(PATH_RAW_CURSOS, PATH_RAW_SEDAP)

# 2. Exibição e Informações Básicas
display(dim_curso.head(10))
dim_curso.info()

print("\nValores Nulos:\n", dim_curso.isna().sum())
print("\nRegistros Duplicados:", dim_curso.duplicated(subset=["CO_CURSO"]).sum())

# Asserts de segurança
assert dim_curso["CO_CURSO"].is_unique, "O CO_CURSO da dimensão curso deve ser único"

# 3. Exportação para CSV Processado
PATH_DIM_CURSO = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_curso_ufpb_campus_1.csv"
PATH_DIM_CURSO.parent.mkdir(parents=True, exist_ok=True)

dim_curso.to_csv(
    PATH_DIM_CURSO,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print(f" Dimensão Curso salva com sucesso em: {PATH_DIM_CURSO}")

,CO_CURSO,NO_CURSO,CO_CINE_ROTULO,DS_GRAU_ACADEMICO,DS_MODALIDADE_ENSINO,NO_MUNICIPIO
0,1203263,ADMINISTRAÇÃO PÚBLICA,0413A02,Bacharelado,EaD,João Pessoa
1,1203266,COMPUTAÇÃO,0114C05,Licenciatura,EaD,João Pessoa
2,1261913,LETRAS - ESPANHOL,0115L02,Licenciatura,EaD,João Pessoa
3,1261910,LETRAS - INGLÊS,0115L04,Licenciatura,EaD,João Pessoa
4,109954,LETRAS - LÍNGUA PORTUGUESA,0115L13,Licenciatura,EaD,João Pessoa
5,1126690,LETRAS - LÍNGUA PORTUGUESA E LIBRAS,0115L18,Licenciatura,EaD,João Pessoa
6,109948,MATEMÁTICA,0114M01,Licenciatura,EaD,João Pessoa
7,109950,PEDAGOGIA,0113P01,Licenciatura,EaD,João Pessoa
8,13395,ADMINISTRAÇÃO,0413A01,Bacharelado,Presencial,João Pessoa
9,1127907,ALIMENTOS,0721A01,Tecnológico,Presencial,João Pessoa


<class 'pandas.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   CO_CURSO              94 non-null     str  
 1   NO_CURSO              94 non-null     str  
 2   CO_CINE_ROTULO        94 non-null     str  
 3   DS_GRAU_ACADEMICO     94 non-null     str  
 4   DS_MODALIDADE_ENSINO  94 non-null     str  
 5   NO_MUNICIPIO          94 non-null     str  
dtypes: str(6)
memory usage: 10.4 KB

Valores Nulos:
 CO_CURSO                0
NO_CURSO                0
CO_CINE_ROTULO          0
DS_GRAU_ACADEMICO       0
DS_MODALIDADE_ENSINO    0
NO_MUNICIPIO            0
dtype: int64

Registros Duplicados: 0
 Dimensão Curso salva com sucesso em: c:\Users\reisc\Desktop\Projeto-Analise-de-Dados-main\data\processed\Dimensões\dim_curso_ufpb_campus_1.csv


In [7]:
# Validação de divergências entre INEP e SEDAP
df_sedap = pd.read_csv(PATH_RAW_SEDAP, sep=None, engine="python", encoding="utf-8-sig")
df_censo = pd.read_csv(PATH_RAW_CURSOS, sep=";", encoding="latin1", dtype=str)

df_campus_1 = df_censo[(df_censo["CO_IES"] == "579") & (df_censo["CO_MUNICIPIO"] == "2507507")]

codigos_inep = set(df_campus_1["CO_CURSO"].astype(str).str.strip())
codigos_sedap = set(df_sedap["CO_CURSO"].astype(str).str.strip())

codigos_faltantes = codigos_inep - codigos_sedap
cursos_fora = df_campus_1[df_campus_1["CO_CURSO"].astype(str).str.strip().isin(codigos_faltantes)]

print(f" Cursos no Censo/INEP não encontrados no SEDAP: {len(cursos_fora)}")
display(cursos_fora[["CO_CURSO", "NO_CURSO", "TP_GRAU_ACADEMICO", "TP_MODALIDADE_ENSINO"]])

 Cursos no Censo/INEP não encontrados no SEDAP: 2


,CO_CURSO,NO_CURSO,TP_GRAU_ACADEMICO,TP_MODALIDADE_ENSINO
153984,22457,Comunicação Social - Radialismo,1,1
154035,26565,Psicologia,2,1


### Cruzamento de Dados (SEDAP+ x INEP) e Truncamento de Históricos

Realizamos o cruzamento via `INNER JOIN` entre a base do SEDAP+ e o Censo do INEP para unificar os códigos do INEP com a nomenclatura oficial dos cursos.

> **Nota de Validação e Regra de Negócio:**
> * **Censo INEP (Campus I):** 96 cursos cadastrados.
> * **Base Tratada (SEDAP+):** 94 cursos ativos.
>
> **Justificativa da Divergência (-2 Cursos):**
> Os cursos abaixo constam na base do Censo INEP, porém não possuem registros ativos ou movimentação na base recente do SEDAP+:
> 1. `CO_CURSO: 153984` — **Comunicação Social - Radialismo** *(código de matriz antiga)*
> 2. `CO_CURSO: 154035` — **Psicologia** *(código de matriz antiga)*
>
> Optou-se pela manutenção do `INNER JOIN` para garantir a integridade referencial da **Tabela Fato Assistência Estudantil**, mantendo na dimensão apenas cursos com histórico de atendimento e matrículas ativas no sistema de concessão de bolsas.

## 2. Dimensão Centro
### 2.1 Gerar e validar Dataframe

In [59]:
from src.dimensions.dim_centro import criar_dim_centro

# Instancia a dimensão
dim_centro = criar_dim_centro()

# Visualização e validações
display(dim_centro)
dim_centro.info()
print("Valores nulos:\n", dim_centro.isna().sum())
print("\nDuplicados:", dim_centro.duplicated().sum())

# Validações via Assert
assert len(dim_centro) == 13, "A dimensão centro deve conter 13 registros"
assert dim_centro["ID_CENTRO"].is_unique, "Os IDs dos centros devem ser únicos"

,ID_CENTRO,CENTRO
0,1,CCEN
1,2,CCHLA
2,3,CCTA
3,4,CCS
4,5,CCSA
5,6,CE
6,7,CT
7,8,CCJ
8,9,CBIOTEC
9,10,CCM


<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   ID_CENTRO  13 non-null     int64
 1   CENTRO     13 non-null     str  
dtypes: int64(1), str(1)
memory usage: 387.0 bytes
Valores nulos:
 ID_CENTRO    0
CENTRO       0
dtype: int64

Duplicados: 0


### 2.2 Exportação para CSV

In [60]:
# Exportação para pasta de dimensões processadas
PATH_DIM_CENTRO = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_centro.csv"
PATH_DIM_CENTRO.parent.mkdir(parents=True, exist_ok=True)

dim_centro.to_csv(
    PATH_DIM_CENTRO,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print(f"✅ Dimensão Centro salva com sucesso em: {PATH_DIM_CENTRO}")

✅ Dimensão Centro salva com sucesso em: c:\Users\reisc\Desktop\Projeto-Analise-de-Dados-main\data\processed\Dimensões\dim_centro.csv


## 3.0 Dimensão Curso-Centro

In [69]:
# Importação e recarregamento do módulo
import importlib
from src.dimensions.dim_curso_centro import criar_curso_centro
import src.dimensions.dim_curso_centro as dcc

importlib.reload(dcc)

# Instancia a tabela curso_centro
# Passe o dataframe de cursos gerado na etapa anterior (ex: dim_curso_ufpb_campus_1)
curso_centro = criar_curso_centro(dim_curso_campus_1, dim_centro)

# Exibe as primeiras linhas e métricas de validação
display(curso_centro.head())
curso_centro.info()

print("Valores Nulos:\n", curso_centro.isna().sum())
print("\nRegistros Duplicados:", curso_centro.duplicated().sum())

,CO_CURSO,NO_CURSO,ID_CENTRO
0,1203263,ADMINISTRAÇÃO PÚBLICA,5
1,1203266,COMPUTAÇÃO,11
2,1261913,LETRAS - ESPANHOL,2
3,1261910,LETRAS - INGLÊS,2
4,109954,LETRAS - LÍNGUA PORTUGUESA,2


<class 'pandas.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   CO_CURSO   94 non-null     str  
 1   NO_CURSO   94 non-null     str  
 2   ID_CENTRO  94 non-null     Int64
dtypes: Int64(1), str(2)
memory usage: 4.6 KB
Valores Nulos:
 CO_CURSO     0
NO_CURSO     0
ID_CENTRO    0
dtype: int64

Registros Duplicados: 0


In [70]:
# Valida se cada curso aparece apenas uma vez na dimensão
assert curso_centro["CO_CURSO"].is_unique, "Atenção: Existem códigos de curso duplicados!"
print(" Validação concluída: Todos os códigos de curso são únicos.")

 Validação concluída: Todos os códigos de curso são únicos.


In [71]:
# Exportação para a pasta de dimensões processadas
PATH_CURSO_CENTRO = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_curso_centro.csv"
PATH_CURSO_CENTRO.parent.mkdir(parents=True, exist_ok=True)

curso_centro.to_csv(
    PATH_CURSO_CENTRO,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print(f"✅ Dimensão Curso-Centro salva com sucesso em: {PATH_CURSO_CENTRO}")

✅ Dimensão Curso-Centro salva com sucesso em: c:\Users\reisc\Desktop\Projeto-Analise-de-Dados-main\data\processed\Dimensões\dim_curso_centro.csv


## 3. Dimensão Sexo

In [77]:
import importlib
import src.dimensions.dim_sexo as ds

importlib.reload(ds)
from src.dimensions.dim_sexo import criar_dim_sexo

dim_sexo = criar_dim_sexo()
dim_sexo.to_csv(BASE_DIR / "data/processed/Dimensões/dim_sexo.csv", index=False, sep=";", encoding="utf-8-sig")

display(dim_sexo)

,ID_SEXO,CD_SEXO,DESCRICAO
0,1,M,Masculino
1,2,F,Feminino


## 4. Dimensão Raça

In [78]:
import importlib
import src.dimensions.dim_raca as dr

importlib.reload(dr)
from src.dimensions.dim_raca import criar_dim_raca

# Instancia a dimensão estática
dim_raca = criar_dim_raca()

# Visualização e validações
display(dim_raca)
dim_raca.info()

print("Valores Nulos:\n", dim_raca.isna().sum())
print("\nRegistros Duplicados:", dim_raca.duplicated().sum())

# Asserts de segurança
assert len(dim_raca) == 6, "A dimensão raça deve conter exatamente 6 categorias oficiais"
assert dim_raca["ID_RACA"].is_unique, "Os IDs da dimensão raça devem ser únicos"

,ID_RACA,DESCRICAO
0,0,Não informado
1,1,Branca
2,2,Preta
3,3,Parda
4,4,Amarela
5,5,Indígena


<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   ID_RACA    6 non-null      int64
 1   DESCRICAO  6 non-null      str  
dtypes: int64(1), str(1)
memory usage: 274.0 bytes
Valores Nulos:
 ID_RACA      0
DESCRICAO    0
dtype: int64

Registros Duplicados: 0


In [79]:
PATH_DIM_RACA = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_raca.csv"
PATH_DIM_RACA.parent.mkdir(parents=True, exist_ok=True)

dim_raca.to_csv(
    PATH_DIM_RACA,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print(f"✅ Dimensão Raça salva com sucesso em: {PATH_DIM_RACA}")

✅ Dimensão Raça salva com sucesso em: c:\Users\reisc\Desktop\Projeto-Analise-de-Dados-main\data\processed\Dimensões\dim_raca.csv


## 5. Dimensão Turno (adição de "Não informado")

Existem cursos **EaD sem turno preenchido** no SEDAP+. Para não descartar
esses registros na fato, criamos aqui o registro `ID_TURNO = 0` ->
`"Não informado"`.

Na tabela fato, os valores nulos de turno serão substituídos por
`ID_TURNO = 0` (ver notebook da Fato, etapa "Atualizar a dimensão turno").

In [80]:
import importlib
import src.dimensions.dim_turno as dt

importlib.reload(dt)
from src.dimensions.dim_turno import criar_dim_turno

# Instancia a dimensão estática
dim_turno = criar_dim_turno()

# Visualização e validações
display(dim_turno)
dim_turno.info()

print("Valores Nulos:\n", dim_turno.isna().sum())
print("\nRegistros Duplicados:", dim_turno.duplicated().sum())

# Asserts de segurança
assert len(dim_turno) == 5, "A dimensão turno deve conter exatamente 5 registros (0 a 4)"
assert dim_turno["ID_TURNO"].is_unique, "Os IDs da dimensão turno devem ser únicos"

,ID_TURNO,CD_TURNO,DESCRICAO
0,0,NI,Não informado
1,1,MAT,Matutino
2,2,VESP,Vespertino
3,3,NOT,Noturno
4,4,INT,Integral


<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   ID_TURNO   5 non-null      int64
 1   CD_TURNO   5 non-null      str  
 2   DESCRICAO  5 non-null      str  
dtypes: int64(1), str(2)
memory usage: 314.0 bytes
Valores Nulos:
 ID_TURNO     0
CD_TURNO     0
DESCRICAO    0
dtype: int64

Registros Duplicados: 0


In [81]:
PATH_DIM_TURNO = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_turno.csv"
PATH_DIM_TURNO.parent.mkdir(parents=True, exist_ok=True)

dim_turno.to_csv(
    PATH_DIM_TURNO,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print(f"✅ Dimensão Turno salva com sucesso em: {PATH_DIM_TURNO}")

✅ Dimensão Turno salva com sucesso em: c:\Users\reisc\Desktop\Projeto-Analise-de-Dados-main\data\processed\Dimensões\dim_turno.csv


## 6. Dimensão Grau

Derivada da `dim_curso` já filtrada para o Campus I (etapa 2).

In [83]:
import importlib
import src.dimensions.dim_grau as dg

importlib.reload(dg)
from src.dimensions.dim_grau import criar_dim_grau

# Instancia a dimensão passando a dim_curso
dim_grau = criar_dim_grau(dim_curso)

# Visualização e validações
display(dim_grau)
dim_grau.info()

print("Valores Nulos:\n", dim_grau.isna().sum())
print("\nRegistros Duplicados:", dim_grau.duplicated().sum())

# Asserts de segurança
assert dim_grau["ID_GRAU"].is_unique, "Os IDs da dimensão grau acadêmico devem ser únicos"

,ID_GRAU,DESCRICAO
0,1,Bacharelado
1,2,Licenciatura
2,3,Tecnológico


<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   ID_GRAU    3 non-null      int64
 1   DESCRICAO  3 non-null      str  
dtypes: int64(1), str(1)
memory usage: 216.0 bytes
Valores Nulos:
 ID_GRAU      0
DESCRICAO    0
dtype: int64

Registros Duplicados: 0


In [84]:
PATH_DIM_GRAU = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_grau.csv"
PATH_DIM_GRAU.parent.mkdir(parents=True, exist_ok=True)

dim_grau.to_csv(
    PATH_DIM_GRAU,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print(f"✅ Dimensão Grau Acadêmico salva com sucesso em: {PATH_DIM_GRAU}")

✅ Dimensão Grau Acadêmico salva com sucesso em: c:\Users\reisc\Desktop\Projeto-Analise-de-Dados-main\data\processed\Dimensões\dim_grau.csv


## 7. Dimensão Modalidade

Também derivada da `dim_curso` já filtrada para o Campus I (etapa 2).

In [87]:
import importlib
import src.dimensions.dim_modalidade as dm

importlib.reload(dm)
from src.dimensions.dim_modalidade import criar_dim_modalidade

# Instancia a dimensão passando dim_curso (ou sem argumentos se optar pela estática)
dim_modalidade = criar_dim_modalidade()

# Visualização e validações
display(dim_modalidade)
dim_modalidade.info()

print("Valores Nulos:\n", dim_modalidade.isna().sum())
print("\nRegistros Duplicados:", dim_modalidade.duplicated().sum())

# Asserts de segurança
assert dim_modalidade["ID_MODALIDADE"].is_unique, "Os IDs da dimensão modalidade devem ser únicos"

,ID_MODALIDADE,CD_MODALIDADE,DESCRICAO
0,1,PRES,Presencial
1,2,EAD,Educação a Distância


<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   ID_MODALIDADE  2 non-null      int64
 1   CD_MODALIDADE  2 non-null      str  
 2   DESCRICAO      2 non-null      str  
dtypes: int64(1), str(2)
memory usage: 220.0 bytes
Valores Nulos:
 ID_MODALIDADE    0
CD_MODALIDADE    0
DESCRICAO        0
dtype: int64

Registros Duplicados: 0


In [89]:
PATH_DIM_MODALIDADE = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_modalidade.csv"
PATH_DIM_MODALIDADE.parent.mkdir(parents=True, exist_ok=True)

dim_modalidade.to_csv(
    PATH_DIM_MODALIDADE,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print(f"Dimensão Modalidade salva com sucesso em: {PATH_DIM_MODALIDADE}")

Dimensão Modalidade salva com sucesso em: c:\Users\reisc\Desktop\Projeto-Analise-de-Dados-main\data\processed\Dimensões\dim_modalidade.csv


## Construção da dimensão auxilio

In [90]:
import importlib
import src.dimensions.dim_auxilio as da

importlib.reload(da)
from src.dimensions.dim_auxilio import criar_dim_auxilio

# Instancia a dimensão estática
dim_auxilio = criar_dim_auxilio()

# Visualização e validação
display(dim_auxilio)
dim_auxilio.info()

print("Valores Nulos:\n", dim_auxilio.isna().sum())
print("\nRegistros Duplicados:", dim_auxilio.duplicated().sum())

# Asserts de segurança
assert len(dim_auxilio) == 6, "A dimensão auxílio deve conter exatamente 6 categorias"
assert dim_auxilio["ID_AUXILIO"].is_unique, "Os IDs da dimensão auxílio devem ser únicos"

,ID_AUXILIO,TIPO_AUXILIO,DESCRICAO
0,1,IN_APOIO_ALIMENTACAO,Auxílio Alimentação / RU
1,2,IN_APOIO_MORADIA,Auxílio Moradia
2,3,IN_APOIO_TRANSPORTE,Auxílio Transporte
3,4,IN_APOIO_MATERIAL_DIDATICO,Auxílio Material Didático
4,5,IN_APOIO_BOLSA_PERMANENCIA,Bolsa Permanência
5,6,IN_APOIO_BOLSA_TRABALHO,Bolsa Trabalho


<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   ID_AUXILIO    6 non-null      int64
 1   TIPO_AUXILIO  6 non-null      str  
 2   DESCRICAO     6 non-null      str  
dtypes: int64(1), str(2)
memory usage: 527.0 bytes
Valores Nulos:
 ID_AUXILIO      0
TIPO_AUXILIO    0
DESCRICAO       0
dtype: int64

Registros Duplicados: 0


In [91]:
PATH_DIM_AUXILIO = BASE_DIR / "data" / "processed" / "Dimensões" / "dim_auxilio.csv"
PATH_DIM_AUXILIO.parent.mkdir(parents=True, exist_ok=True)

dim_auxilio.to_csv(
    PATH_DIM_AUXILIO,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print(f"✅ Dimensão Auxílio salva com sucesso em: {PATH_DIM_AUXILIO}")

✅ Dimensão Auxílio salva com sucesso em: c:\Users\reisc\Desktop\Projeto-Analise-de-Dados-main\data\processed\Dimensões\dim_auxilio.csv
